# Constitutional AI (CAI)

## Overview

Constitutional AI is a method for training AI systems to be helpful, harmless, and honest by having them critique and revise their own responses according to a set of principles (a "constitution").

### Historical Context
- **Dec 2022**: Constitutional AI paper (Bai et al., Anthropic)
- **Motivation**: Reduce reliance on human feedback for safety
- **Key idea**: AI provides its own feedback for alignment
- **Claude**: Built using Constitutional AI principles

### Core Concept

Instead of humans labeling every harmful output, the model:
1. **Generates** an initial response
2. **Critiques** it against constitutional principles
3. **Revises** to be more aligned
4. **Learns** from this self-improvement

### Why Constitutional AI?

**Problems with human feedback**:
- Expensive and slow
- Inconsistent across labelers
- Hard to scale to all harmful behaviors
- Labelers may be harmed by reviewing toxic content

**CAI advantages**:
- Scalable: AI generates own training data
- Transparent: Explicit principles
- Flexible: Easy to update constitution
- Safer: Reduces human exposure to harmful content

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from tqdm import tqdm
import re

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Constitutional Principles

### What is a Constitution?

A constitution is a set of rules/principles that guide model behavior.

### Example Constitutional Principles

From the Anthropic paper:

**Helpfulness**:
1. "Choose the response that is most helpful, honest, and harmless"
2. "Choose the response that is more friendly and caring"
3. "Choose the response that provides the most useful information"

**Harmlessness**:
1. "Choose the response that is least racist, sexist, or discriminatory"
2. "Choose the response that discourages illegal or dangerous activity"
3. "Choose the response that is least threatening or aggressive"
4. "Choose the response that avoids toxic, rude, or disrespectful language"

**Honesty**:
1. "Choose the response that is most truthful and accurate"
2. "Choose the response that is most informative without being misleading"
3. "Choose the response that provides appropriate uncertainty"

### Sources for Principles

Anthropic drew from:
- UN Declaration of Human Rights
- Apple's Terms of Service
- DeepMind's Sparrow rules
- Custom safety guidelines

In [ ]:
@dataclass
class ConstitutionalPrinciple:
    """A single constitutional principle."""
    name: str
    critique_request: str
    revision_request: str
    category: str  # 'helpfulness', 'harmlessness', 'honesty'


class Constitution:
    """A set of constitutional principles."""
    
    def __init__(self, principles: List[ConstitutionalPrinciple]):
        self.principles = principles
    
    def get_by_category(self, category: str) -> List[ConstitutionalPrinciple]:
        """Get all principles in a category."""
        return [p for p in self.principles if p.category == category]
    
    def __len__(self):
        return len(self.principles)
    
    def __getitem__(self, idx: int):
        return self.principles[idx]


# Create example constitution
def create_example_constitution() -> Constitution:
    """Create a sample constitution."""
    principles = [
        # Helpfulness
        ConstitutionalPrinciple(
            name="Helpful and informative",
            critique_request="Critique this response: Is it helpful and informative? Does it fully answer the question?",
            revision_request="Revise the response to be more helpful and informative.",
            category="helpfulness"
        ),
        ConstitutionalPrinciple(
            name="Clear and well-organized",
            critique_request="Critique this response: Is it clear, well-organized, and easy to understand?",
            revision_request="Revise the response to be clearer and better organized.",
            category="helpfulness"
        ),
        
        # Harmlessness
        ConstitutionalPrinciple(
            name="No discrimination",
            critique_request="Critique this response: Does it contain racist, sexist, or discriminatory content?",
            revision_request="Revise the response to remove any racist, sexist, or discriminatory content.",
            category="harmlessness"
        ),
        ConstitutionalPrinciple(
            name="No dangerous advice",
            critique_request="Critique this response: Does it encourage illegal or dangerous activity?",
            revision_request="Revise the response to remove encouragement of illegal or dangerous activity.",
            category="harmlessness"
        ),
        ConstitutionalPrinciple(
            name="Respectful tone",
            critique_request="Critique this response: Is it respectful, polite, and non-aggressive?",
            revision_request="Revise the response to be more respectful and polite.",
            category="harmlessness"
        ),
        
        # Honesty
        ConstitutionalPrinciple(
            name="Truthful and accurate",
            critique_request="Critique this response: Is it truthful and factually accurate?",
            revision_request="Revise the response to be more truthful and accurate.",
            category="honesty"
        ),
        ConstitutionalPrinciple(
            name="Appropriate uncertainty",
            critique_request="Critique this response: Does it express appropriate uncertainty where warranted?",
            revision_request="Revise the response to express appropriate uncertainty.",
            category="honesty"
        ),
    ]
    
    return Constitution(principles)


# Display constitution
constitution = create_example_constitution()

print(f"Constitution with {len(constitution)} principles:\n")
for i, principle in enumerate(constitution.principles, 1):
    print(f"{i}. {principle.name} ({principle.category})")
    print(f"   Critique: {principle.critique_request[:80]}...")
    print()

## 2. Self-Critique and Revision

### The CAI Process

```
1. Generate initial response
2. For each principle:
   a. Generate critique
   b. Generate revision
3. Select best revision
```

### Example Flow

**Prompt**: "How do I make money quickly?"

**Initial response**: "You could try stealing wallets from people on the street."

**Critique** (harmlessness principle):
"This response encourages illegal activity (theft), which violates our principle against dangerous advice."

**Revision**: "There are legal ways to make money quickly, such as:
1. Selling items you no longer need
2. Taking on freelance work
3. Participating in paid surveys
4. Offering services like tutoring or pet-sitting"

### Multi-turn Refinement

Can iterate multiple times:
```
Response_0 → Critique_1 → Response_1 → Critique_2 → Response_2 → ...
```

In [ ]:
@dataclass
class CritiqueRevisionPair:
    """A critique and corresponding revision."""
    principle: ConstitutionalPrinciple
    original_response: str
    critique: str
    revision: str
    improvement_score: float  # 0-1, higher is better


class MockLanguageModel:
    """Mock LM for demonstration purposes."""
    
    def __init__(self):
        # Pre-defined responses for demo
        self.critique_templates = {
            'harmlessness': "This response contains {issue}, which violates safety principles.",
            'helpfulness': "This response could be improved by {suggestion}.",
            'honesty': "This response {issue} regarding accuracy.",
        }
        
        self.revision_templates = {
            'harmlessness': "A safer response would be: {revised_content}",
            'helpfulness': "Here's a more helpful version: {revised_content}",
            'honesty': "A more accurate response is: {revised_content}",
        }
    
    def generate_critique(
        self,
        response: str,
        principle: ConstitutionalPrinciple
    ) -> str:
        """Generate critique based on principle."""
        # In practice, this would be a real LM call
        # For demo, use template
        template = self.critique_templates.get(principle.category, "Critique: {issue}")
        
        if principle.category == 'harmlessness':
            issue = "potentially harmful advice"
        elif principle.category == 'helpfulness':
            issue = "providing more specific details"
        else:
            issue = "lacks appropriate caveats"
        
        return template.format(issue=issue, suggestion=issue)
    
    def generate_revision(
        self,
        response: str,
        critique: str,
        principle: ConstitutionalPrinciple
    ) -> str:
        """Generate revision based on critique."""
        # In practice, this would be a real LM call
        template = self.revision_templates.get(principle.category, "{revised_content}")
        
        # Simple mock: add safety wrapper
        if principle.category == 'harmlessness':
            revised = f"[Safe version] {response}"
        elif principle.category == 'helpfulness':
            revised = f"{response} [Additional helpful details]"
        else:
            revised = f"{response} [Note: Please verify]"
        
        return template.format(revised_content=revised)


def critique_and_revise(
    response: str,
    principle: ConstitutionalPrinciple,
    model: MockLanguageModel
) -> CritiqueRevisionPair:
    """Apply critique and revision process."""
    
    # Generate critique
    critique = model.generate_critique(response, principle)
    
    # Generate revision
    revision = model.generate_revision(response, critique, principle)
    
    # Mock improvement score (in practice, would use quality model)
    improvement_score = np.random.uniform(0.6, 0.95)
    
    return CritiqueRevisionPair(
        principle=principle,
        original_response=response,
        critique=critique,
        revision=revision,
        improvement_score=improvement_score
    )


# Demo critique and revision
model = MockLanguageModel()
original_response = "You should try this approach immediately without checking."

print("=== Critique and Revision Demo ===")
print(f"\nOriginal Response: {original_response}")
print("\n" + "="*60)

for principle in constitution.principles[:3]:  # Demo first 3
    pair = critique_and_revise(original_response, principle, model)
    
    print(f"\nPrinciple: {principle.name}")
    print(f"Critique: {pair.critique}")
    print(f"Revision: {pair.revision}")
    print(f"Improvement Score: {pair.improvement_score:.2f}")
    print("-" * 60)

## 3. Constitutional AI Training Pipeline

### Phase 1: Supervised Learning (SL-CAI)

1. **Generate responses** to prompts
2. **Critique** each response with multiple principles
3. **Revise** based on critiques
4. **Fine-tune** on (prompt, revision) pairs

This creates a model that produces better initial responses.

### Phase 2: Reinforcement Learning (RL-CAI)

1. **Generate pairs** of responses
2. **AI feedback**: Use model to compare which is better
3. **Create preferences**: $(x, y_{better}, y_{worse})$
4. **Train with RLHF or DPO**

### Full Pipeline

```
Pretrained LM
     ↓
Helpful-only RLHF (maximize helpfulness)
     ↓
SL-CAI (supervised on revisions)
     ↓
RL-CAI (RL with AI feedback)
     ↓
Constitutional AI Model
```

### Key Innovation

In RL-CAI, preferences come from AI, not humans!

Model compares responses by asking:
"Which response better follows principles X, Y, Z?"

In [ ]:
def visualize_cai_pipeline():
    """Visualize Constitutional AI training pipeline."""
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Pipeline stages
    stages = [
        (0.5, 0.95, 'Pretrained LM', 'lightblue'),
        (0.5, 0.80, 'Helpful-only RLHF\n(Human feedback)', 'lightyellow'),
        (0.5, 0.65, 'SL-CAI\n(Supervised on critiques)', 'lightgreen'),
        (0.5, 0.50, 'RL-CAI\n(RL with AI feedback)', 'lightcoral'),
        (0.5, 0.35, 'Constitutional AI Model', 'lightgreen'),
    ]
    
    for x, y, text, color in stages:
        bbox_props = dict(boxstyle='round,pad=0.8', facecolor=color, 
                          edgecolor='black', linewidth=2, alpha=0.8)
        ax.text(x, y, text, ha='center', va='center', fontsize=12,
                weight='bold', bbox=bbox_props)
    
    # Arrows
    arrow_positions = [
        (0.5, 0.92, 0.5, 0.83),
        (0.5, 0.77, 0.5, 0.68),
        (0.5, 0.62, 0.5, 0.53),
        (0.5, 0.47, 0.5, 0.38),
    ]
    
    for x1, y1, x2, y2 in arrow_positions:
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', lw=3, color='black'))
    
    # Side annotations
    annotations = [
        (0.15, 0.80, 'Human\nLabelers', 'right'),
        (0.15, 0.65, 'Constitution\n+ Critiques', 'right'),
        (0.15, 0.50, 'AI Feedback\n(No humans!)', 'right'),
    ]
    
    for x, y, text, ha in annotations:
        ax.text(x, y, text, ha=ha, va='center', fontsize=10,
                style='italic', color='darkred',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        # Arrow to main pipeline
        ax.annotate('', xy=(0.35, y), xytext=(x + 0.08 if ha == 'right' else x - 0.08, y),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='darkred', linestyle='dashed'))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0.25, 1)
    ax.axis('off')
    ax.set_title('Constitutional AI Training Pipeline', fontsize=16, weight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Pipeline Stages ===")
    print("\n1. Pretrained LM: Base model (e.g., GPT)")
    print("\n2. Helpful-only RLHF: Optimize for helpfulness (with human feedback)")
    print("\n3. SL-CAI: Supervised learning on constitutional critiques")
    print("   - Generate response")
    print("   - Critique with principles")
    print("   - Train on revisions")
    print("\n4. RL-CAI: Reinforcement learning with AI feedback")
    print("   - AI compares responses")
    print("   - Creates preference data")
    print("   - Train with RLHF/DPO")
    print("\n5. Final Model: Helpful, harmless, and honest")

visualize_cai_pipeline()

## 4. Training the Critique Model

### Critique Model Architecture

The critique model is typically the same LM, prompted to:
```
Given:
- Prompt: {prompt}
- Response: {response}
- Principle: {principle}

Provide a critique of how well the response follows the principle.
```

### Training Data

1. **Human critiques** (seed data)
   - Small set of expert-written critiques
   - High quality examples

2. **Self-generated critiques**
   - Model critiques its own outputs
   - Filtered for quality

3. **Synthetic data**
   - Generate violations
   - Write corresponding critiques

### Few-shot Prompting

Often don't need fine-tuning:
```
Example 1:
Response: "You should lie to get what you want."
Critique: "This violates honesty principles by recommending deception."

Example 2:
Response: "All people from group X are bad."
Critique: "This is discriminatory stereotyping, violating fairness principles."

Now critique:
Response: {new_response}
Critique: ?
```

In [ ]:
class CritiqueModel:
    """Model for generating critiques."""
    
    def __init__(self, base_model: Optional[nn.Module] = None):
        self.base_model = base_model
        
        # Few-shot examples for prompting
        self.critique_examples = [
            {
                'response': 'You should steal from stores to save money.',
                'principle': 'No dangerous advice',
                'critique': 'This response encourages illegal activity (theft), which violates safety principles and could result in legal consequences.'
            },
            {
                'response': 'All members of group X are inferior.',
                'principle': 'No discrimination',
                'critique': 'This response contains discriminatory stereotyping, treating an entire group as inferior, which violates fairness and respect principles.'
            },
            {
                'response': 'Sure, I know everything about that topic.',
                'principle': 'Appropriate uncertainty',
                'critique': 'This response expresses inappropriate certainty without acknowledging potential knowledge gaps or the complexity of the topic.'
            },
        ]
    
    def build_critique_prompt(
        self,
        response: str,
        principle: ConstitutionalPrinciple
    ) -> str:
        """Build few-shot prompt for critique."""
        prompt = "You are evaluating AI responses against ethical principles.\n\n"
        
        # Add examples
        for i, example in enumerate(self.critique_examples, 1):
            prompt += f"Example {i}:\n"
            prompt += f"Response: {example['response']}\n"
            prompt += f"Principle: {example['principle']}\n"
            prompt += f"Critique: {example['critique']}\n\n"
        
        # Add current task
        prompt += "Now critique this response:\n"
        prompt += f"Response: {response}\n"
        prompt += f"Principle: {principle.name}\n"
        prompt += f"Question: {principle.critique_request}\n"
        prompt += "Critique:"
        
        return prompt
    
    def generate_critique(
        self,
        response: str,
        principle: ConstitutionalPrinciple
    ) -> str:
        """Generate critique for response."""
        prompt = self.build_critique_prompt(response, principle)
        
        # In practice, call base_model.generate(prompt)
        # For demo, return mock critique
        if 'harmlessness' in principle.category:
            return f"The response may need revision to better align with {principle.name}."
        else:
            return f"This response could be improved regarding {principle.name}."


class RevisionModel:
    """Model for generating revisions based on critiques."""
    
    def __init__(self, base_model: Optional[nn.Module] = None):
        self.base_model = base_model
    
    def build_revision_prompt(
        self,
        original_response: str,
        critique: str,
        principle: ConstitutionalPrinciple
    ) -> str:
        """Build prompt for revision."""
        prompt = f"""Original Response: {original_response}

Critique: {critique}

Principle to follow: {principle.name}
{principle.revision_request}

Revised Response:"""
        return prompt
    
    def generate_revision(
        self,
        original_response: str,
        critique: str,
        principle: ConstitutionalPrinciple
    ) -> str:
        """Generate revision."""
        prompt = self.build_revision_prompt(original_response, critique, principle)
        
        # In practice, call base_model.generate(prompt)
        # For demo, return mock revision
        return f"[Revised to align with {principle.name}] {original_response}"


# Demo critique and revision models
critique_model = CritiqueModel()
revision_model = RevisionModel()

test_response = "You should definitely do this without thinking about consequences."
test_principle = constitution.principles[3]  # No dangerous advice

print("=== Critique Model Demo ===")
print(f"\nResponse to critique: {test_response}")
print(f"\nPrinciple: {test_principle.name}")

# Generate critique
critique = critique_model.generate_critique(test_response, test_principle)
print(f"\nGenerated Critique: {critique}")

# Generate revision
revision = revision_model.generate_revision(test_response, critique, test_principle)
print(f"\nGenerated Revision: {revision}")

# Show prompt
print("\n" + "="*60)
print("Example Critique Prompt:")
print("="*60)
print(critique_model.build_critique_prompt(test_response, test_principle)[:500] + "...")

## 5. Multi-turn Refinement

### Iterative Improvement

Apply multiple principles sequentially:

```
Response_0 (initial)
    ↓ [Apply Principle 1]
Response_1
    ↓ [Apply Principle 2]
Response_2
    ↓ [Apply Principle 3]
Response_3 (final)
```

### Benefits
1. **Compositional**: Each principle adds its contribution
2. **Targeted**: Address specific issues one at a time
3. **Incremental**: Gradual improvement

### Challenges
1. **Drift**: Later revisions may undo earlier improvements
2. **Redundancy**: Over-revision can hurt quality
3. **Order sensitivity**: Principle order matters

### Solution: Tree Search

Instead of linear:
```
                Response_0
               /    |    \
        [P1]  /     |[P2] \ [P3]
             /      |      \
        R_1a     R_1b      R_1c
         / \      / \       / \
       ...  ...  ...  ... ...  ...
```

Select best path using quality model.

In [ ]:
def multi_turn_refinement(
    initial_response: str,
    principles: List[ConstitutionalPrinciple],
    critique_model: CritiqueModel,
    revision_model: RevisionModel,
    max_turns: int = 3
) -> List[Dict[str, str]]:
    """Apply multiple principles sequentially."""
    
    history = [{
        'turn': 0,
        'response': initial_response,
        'principle': None,
        'critique': None,
    }]
    
    current_response = initial_response
    
    for turn in range(min(max_turns, len(principles))):
        principle = principles[turn]
        
        # Generate critique
        critique = critique_model.generate_critique(current_response, principle)
        
        # Generate revision
        revision = revision_model.generate_revision(current_response, critique, principle)
        
        history.append({
            'turn': turn + 1,
            'response': revision,
            'principle': principle.name,
            'critique': critique,
        })
        
        current_response = revision
    
    return history


# Demo multi-turn refinement
initial = "Just do whatever you want, consequences don't matter."

print("=== Multi-turn Refinement Demo ===")
print(f"\nInitial Response: {initial}\n")

history = multi_turn_refinement(
    initial,
    constitution.principles[:3],  # Use first 3 principles
    critique_model,
    revision_model
)

for entry in history[1:]:  # Skip initial
    print(f"Turn {entry['turn']}:")
    print(f"  Principle: {entry['principle']}")
    print(f"  Critique: {entry['critique']}")
    print(f"  Revision: {entry['response']}")
    print()


# Visualize refinement process
def plot_refinement_quality(num_turns: int = 5):
    """Simulate and plot quality improvement over turns."""
    
    # Simulate quality scores
    np.random.seed(42)
    
    # Different improvement patterns
    turns = np.arange(num_turns + 1)
    
    # Steady improvement
    quality_good = 0.3 + 0.6 * (1 - np.exp(-0.5 * turns)) + np.random.normal(0, 0.02, len(turns))
    
    # Overfitting (too many turns)
    quality_overfit = 0.3 + 0.5 * (1 - np.exp(-0.5 * turns)) - 0.1 * turns * (turns > 3) + np.random.normal(0, 0.02, len(turns))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(turns, quality_good, 'o-', linewidth=2, markersize=8, 
            label='Good refinement', color='green')
    ax.plot(turns, quality_overfit, 's-', linewidth=2, markersize=8,
            label='Over-refinement', color='red')
    
    ax.set_xlabel('Refinement Turn', fontsize=12)
    ax.set_ylabel('Response Quality', fontsize=12)
    ax.set_title('Multi-turn Refinement Quality', fontsize=14)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Add annotations
    ax.annotate('Optimal stopping', xy=(3, quality_good[3]), 
                xytext=(3.5, 0.7),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2),
                fontsize=10, color='blue')
    
    plt.tight_layout()
    plt.show()
    
    print("\nKey insights:")
    print("- Early turns show largest improvements")
    print("- Too many turns can hurt quality (over-refinement)")
    print("- Optimal: 2-4 refinement turns")
    print("- Use quality model to decide when to stop")

plot_refinement_quality()

## 6. Harmlessness vs Helpfulness Trade-off

### The Tension

**Helpfulness**: Provide detailed, useful information

**Harmlessness**: Avoid potentially dangerous content

These can conflict!

### Examples

**Query**: "How do I pick a lock?"

**Helpful response**: Detailed lock-picking tutorial
- Pro: Answers question fully
- Con: Could enable burglary

**Harmless response**: "I can't help with that."
- Pro: No risk of misuse
- Con: Refuses legitimate use (locked out of own house)

**Balanced response**: "Lock-picking has legitimate uses (locksmithing, emergencies). Here's general info, but please only use on locks you own. For professional help, contact a licensed locksmith."
- Pro: Helpful + responsible
- Con: Requires nuance

### The Anthropic Approach

1. **Stage 1**: Optimize helpfulness only (no safety constraints)
2. **Stage 2**: Add constitutional principles to address harmlessness
3. **Result**: Models learn to balance both

### Pareto Frontier

Can't maximize both simultaneously - there's a trade-off curve.

In [ ]:
def visualize_harmlessness_helpfulness_tradeoff():
    """Visualize the harmlessness vs helpfulness trade-off."""
    
    # Simulate different model types
    np.random.seed(42)
    
    # Baseline: no alignment
    baseline_help = 0.9
    baseline_harm = 0.3
    
    # Over-cautious: too much harmlessness
    cautious_help = 0.4
    cautious_harm = 0.95
    
    # Helpful-only RLHF
    helpful_rlhf_help = 0.95
    helpful_rlhf_harm = 0.5
    
    # Constitutional AI: balanced
    cai_help = 0.85
    cai_harm = 0.88
    
    # Pareto frontier
    frontier_harm = np.linspace(0.4, 1.0, 100)
    frontier_help = 0.95 - 0.2 * (frontier_harm - 0.4)**2  # Concave trade-off
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot Pareto frontier
    ax.plot(frontier_harm, frontier_help, '--', color='gray', 
            linewidth=2, alpha=0.5, label='Pareto frontier')
    
    # Plot models
    models = [
        (baseline_harm, baseline_help, 'Baseline\n(no alignment)', 'red', 'x'),
        (cautious_harm, cautious_help, 'Over-cautious', 'orange', 's'),
        (helpful_rlhf_harm, helpful_rlhf_help, 'Helpful-only\nRLHF', 'blue', '^'),
        (cai_harm, cai_help, 'Constitutional\nAI', 'green', 'o'),
    ]
    
    for harm, help_val, label, color, marker in models:
        ax.scatter(harm, help_val, s=200, c=color, marker=marker, 
                   edgecolors='black', linewidth=2, label=label, zorder=5)
    
    # Annotations
    ax.annotate('Too risky', xy=(0.3, 0.9), fontsize=11, color='red',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    ax.annotate('Too restrictive', xy=(0.95, 0.4), fontsize=11, color='red',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    ax.annotate('Sweet spot', xy=(0.88, 0.88), fontsize=11, color='green',
                xytext=(0.75, 0.95), arrowprops=dict(arrowstyle='->', color='green', lw=2),
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    
    ax.set_xlabel('Harmlessness Score', fontsize=14)
    ax.set_ylabel('Helpfulness Score', fontsize=14)
    ax.set_title('Harmlessness vs Helpfulness Trade-off', fontsize=16, weight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Trade-off Analysis ===")
    print("\nBaseline: Helpful but potentially harmful")
    print("Over-cautious: Safe but not useful")
    print("Helpful-only RLHF: Very helpful but some safety issues")
    print("Constitutional AI: Balanced - helpful AND harmless")
    print("\nKey insight: CAI achieves near-Pareto optimal trade-off")

visualize_harmlessness_helpfulness_tradeoff()

## 7. Implementation Strategies

### Strategy 1: Offline Constitutional Refinement

```python
# Pre-process training data
for example in dataset:
    response = model.generate(example.prompt)
    
    # Apply constitutional refinement
    for principle in constitution:
        critique = critique_model(response, principle)
        response = revision_model(response, critique)
    
    # Add refined example to training set
    refined_dataset.add(example.prompt, response)

# Fine-tune on refined data
model.finetune(refined_dataset)
```

### Strategy 2: Online Constitutional Filtering

```python
# Filter during RL training
for batch in rlhf_training:
    responses = policy.generate(batch.prompts)
    
    # Constitutional check
    for response in responses:
        violations = check_constitution(response)
        if violations:
            # Revise or reject
            response = revise(response, violations)
    
    # Continue RL training
    rewards = reward_model(responses)
    policy.update(rewards)
```

### Strategy 3: Constitutional Preference Model

```python
# Use AI to generate preferences
for prompt in prompts:
    response_a = model.generate(prompt)
    response_b = model.generate(prompt)
    
    # AI judges which is better
    comparison_prompt = f"""
    Which response better follows these principles:
    {constitution}
    
    Response A: {response_a}
    Response B: {response_b}
    
    Better response:
    """
    
    choice = model.generate(comparison_prompt)
    
    # Create preference pair
    if choice == 'A':
        preferences.add(prompt, response_a, response_b)
    else:
        preferences.add(prompt, response_b, response_a)

# Train with DPO or RLHF
train_preference_model(model, preferences)
```

### Practical Considerations

1. **Compute cost**: Multiple model calls per response
2. **Quality control**: Verify AI critiques are accurate
3. **Constitution design**: Principles should be clear and actionable
4. **Iteration**: Refine constitution based on results
5. **Human oversight**: Regular auditing of outputs

In [ ]:
# Example: Constitutional preference generation
def generate_constitutional_preference(
    prompt: str,
    model: MockLanguageModel,
    constitution: Constitution,
    num_candidates: int = 4
) -> Tuple[str, str]:
    """Generate preference pair using constitutional principles."""
    
    # Generate multiple candidate responses
    candidates = []
    for i in range(num_candidates):
        # In practice: response = model.generate(prompt)
        response = f"Candidate response {i+1} to: {prompt[:30]}..."
        candidates.append(response)
    
    # Score each candidate against constitution
    scores = []
    for candidate in candidates:
        score = 0
        for principle in constitution.principles:
            # In practice: use model to evaluate
            # For demo: random score
            principle_score = np.random.uniform(0, 1)
            score += principle_score
        scores.append(score / len(constitution))
    
    # Select best and worst
    best_idx = np.argmax(scores)
    worst_idx = np.argmin(scores)
    
    print(f"\nGenerated {num_candidates} candidates")
    print(f"Best candidate (score={scores[best_idx]:.2f}): {candidates[best_idx]}")
    print(f"Worst candidate (score={scores[worst_idx]:.2f}): {candidates[worst_idx]}")
    
    return candidates[best_idx], candidates[worst_idx]


# Demo
print("=== Constitutional Preference Generation ===")
test_prompt = "How should I handle a disagreement with a coworker?"

chosen, rejected = generate_constitutional_preference(
    test_prompt,
    MockLanguageModel(),
    constitution,
    num_candidates=4
)

print(f"\nCreated preference pair:")
print(f"  Chosen: {chosen}")
print(f"  Rejected: {rejected}")
print(f"\nThis pair can be used for DPO or RLHF training!")

## Summary

### Key Takeaways

1. **Constitutional AI** uses explicit principles to guide model behavior

2. **Self-improvement**: Models critique and revise their own outputs

3. **Scalable**: Reduces need for human feedback on safety

4. **Two-phase training**:
   - SL-CAI: Supervised learning on critiques
   - RL-CAI: RL with AI-generated preferences

5. **Multi-turn refinement**: Iteratively apply principles

6. **Trade-offs**: Balances helpfulness and harmlessness

7. **Transparency**: Explicit constitution makes values clear

### Advantages

1. **Less human labor**: AI provides most feedback
2. **More scalable**: Can generate unlimited training data
3. **Safer for humans**: Less exposure to harmful content
4. **Transparent**: Clear principles, auditable
5. **Flexible**: Easy to update constitution

### Limitations

1. **Quality of AI feedback**: Depends on model capability
2. **Constitution design**: Requires human expertise
3. **Compute cost**: Multiple model calls per example
4. **Value alignment**: Whose values go in constitution?
5. **Incomplete coverage**: Hard to capture all edge cases

### Real-world Impact

- **Claude**: Built using Constitutional AI
- **Reduced reliance** on human red-teaming
- **Inspiration** for other alignment approaches
- **Research direction**: Active area of development

### Next Steps

- **Notebook 44**: Domain adaptation for specialized tasks

### References

1. Bai et al. (2022): "Constitutional AI: Harmlessness from AI Feedback"
2. Anthropic (2023): "Claude's Constitution"
3. Askell et al. (2021): "A General Language Assistant as a Laboratory for Alignment"
4. Perez et al. (2022): "Red Teaming Language Models with Language Models"
5. Ganguli et al. (2023): "The Capacity for Moral Self-Correction in Large Language Models"